<a href="https://colab.research.google.com/github/azcsprof/ASU-CSE475-SS25/blob/Unit-4-Lab-1/Unit_4_Lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# STEP 2: Load dataset from from GitHub
url = "https://raw.githubusercontent.com/azcsprof/ASU-CSE475-SS25/Unit-4-Lab-1/AAPL.csv"
df = pd.read_csv(url)

display(df.head())
print("Available columns:", df.columns.tolist())

# Plot Close price trend
plt.figure(figsize=(12, 4))
plt.plot(df['Close'], label='Close Price')
plt.title("Apple Close Price Over Time")
plt.xlabel("Days")
plt.ylabel("Price")
plt.grid(True)
plt.legend()
plt.show()

# Feature Replationships
sns.pairplot(df[['Open', 'High', 'Close', 'Volume']])
plt.suptitle("Feature Relationships", y=1.02)
plt.show()

In [ ]:
X = df[['Open', 'High', 'Volume']]
y = df[['Close']]

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

train_X, test_X, train_y, test_y = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=1)

train_X = torch.tensor(train_X, dtype=torch.float32).unsqueeze(1)
test_X = torch.tensor(test_X, dtype=torch.float32).unsqueeze(1)
train_y = torch.tensor(train_y, dtype=torch.float32)
test_y = torch.tensor(test_y, dtype=torch.float32)

print("Train X shape:", train_X.shape)  # (batch, seq_len=1, features)
print("Train y shape:", train_y.shape)

In [ ]:
class AppleStockDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class StockLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.relu(self.fc1(out))
        return self.fc2(out)

In [ ]:
def train_model(hidden_size, num_layers, learning_rate, batch_size, epochs):
    model = StockLSTM(3, hidden_size, num_layers)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    train_loader = DataLoader(AppleStockDataset(train_X, train_y), batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            preds = model(xb)
            loss = criterion(preds, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")
    return model

In [ ]:
# Mean predictor baseline
baseline_pred = torch.full_like(test_y, train_y.mean())
baseline_mse = nn.MSELoss()(baseline_pred, test_y)
print(f"Baseline (mean-only) MSE: {baseline_mse:.4f}")

In [ ]:
def evaluate_model(model):
    model.eval()
    with torch.no_grad():
        preds = model(test_X)

    mse = nn.MSELoss()(preds, test_y).item()
    print(f"LSTM Model MSE: {mse:.4f}")

    # De-normalize
    actual = scaler_y.inverse_transform(test_y.numpy())
    predicted = scaler_y.inverse_transform(preds.numpy())

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(actual, label="Actual", linewidth=1)
    plt.plot(predicted, label="Predicted", linewidth=1)
    plt.title(f"De-normalized Prediction vs Actual (MSE: {mse:.4f})")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# EVALUATION CELL: Predict, Score, Plot
from sklearn.metrics import r2_score

def evaluate_model(model):
    model.eval()
    with torch.no_grad():
        preds = model(test_X)

    # Compute and expose required r2score
    global r2score
    r2score = r2_score(test_y.numpy(), preds.numpy())
    print(f"R² score: {r2score:.4f}")

    # Plot predictions vs actual
    plt.figure(figsize=(10, 4))
    plt.plot(test_y.numpy(), label='Actual')
    plt.plot(preds.numpy(), label='Predicted')
    plt.title("Predicted vs Actual Close Prices")
    plt.legend()
    plt.grid(True)
    plt.show()

# Train and evaluate
model = train_model(hidden_size=64, num_layers=2, learning_rate=0.001, batch_size=32, epochs=20)
evaluate_model(model)
print("Final R² score for submission:", r2score)

In [ ]:
# 🔍 STEP 1: Day-by-Day Sample Prediction Walkthrough
# Show how the trained model makes predictions on actual input examples
sample_indices = [0, 1, 2, 3, 4]  # You can change these
sample_X = train_X[sample_indices]
sample_y = train_y[sample_indices]

model.eval()
with torch.no_grad():
    sample_preds = model(sample_X)

for i in range(len(sample_indices)):
    features = scaler_X.inverse_transform(sample_X[i].squeeze().numpy().reshape(1, -1))[0]
    true_price = scaler_y.inverse_transform(sample_y[i].numpy().reshape(1, -1))[0][0]
    pred_price = scaler_y.inverse_transform(sample_preds[i].numpy().reshape(1, -1))[0][0]

    print(f"📅 Day {i+1}")
    print(f"   Inputs (Open, High, Volume): {features}")
    print(f"   True Close:  {true_price:.2f}")
    print(f"   Predicted:   {pred_price:.2f}")
    print("-" * 40)


### 🧠 Instructional Note
In this section, we show how the model **gradually learns** to predict the closing price.

- We only train on a few batches per epoch (for speed)
- After each epoch, we **predict 50 samples** and **de-normalize** them
- The plot shows predictions at epochs 0, 5, 10, 15, and 19

**You should see the predictions getting closer to the true values (purple dashed line)** as training progresses.

In [ ]:
# 📈 STEP 2 (FIXED): See How Model Predictions Improve with Training
# Now with real training and proper de-normalization for visibility
snapshots = []
true_vals = scaler_y.inverse_transform(test_y[:50].numpy()).squeeze()

model_snap = StockLSTM(3, 64, 2)
optimizer_snap = torch.optim.Adam(model_snap.parameters(), lr=0.001)
criterion_snap = nn.MSELoss()

train_loader = DataLoader(AppleStockDataset(train_X, train_y), batch_size=32, shuffle=True)

# Train over 20 epochs, using a few batches per epoch
for epoch in range(20):
    model_snap.train()
    for i, (xb, yb) in enumerate(train_loader):
        preds = model_snap(xb)
        loss = criterion_snap(preds, yb)
        optimizer_snap.zero_grad()
        loss.backward()
        optimizer_snap.step()
        pass

    model_snap.eval()
    with torch.no_grad():
        pred_epoch = model_snap(test_X[:50]).numpy()
        pred_epoch = scaler_y.inverse_transform(pred_epoch)
        snapshots.append(pred_epoch.squeeze())

# Plot predictions at several epochs vs true values
plt.figure(figsize=(12, 6))
for i, snapshot in enumerate(snapshots):
    if i % 5 == 0 or i == len(snapshots) - 1:
        plt.plot(snapshot, label=f'Epoch {i}')
plt.plot(true_vals, label='True', linewidth=2, linestyle='--', color='purple')
plt.title("How Model Predictions Improve Over Epochs")
plt.xlabel("Sample Index")
plt.ylabel("Close Price")
plt.grid(True)
plt.legend()
plt.show()


This chart shows how your model’s predictions evolve over time as it trains.

*   At early epochs, predictions are typically close to a constant or average value. This reflects the model’s untrained state — it hasn’t yet learned to connect input patterns to output values.
*   As training progresses, the prediction lines should start to resemble the true values (purple dashed line). You may see the model begin to track upward and downward trends, even if it doesn’t match the exact numbers.

If your predictions stay flat or incorrect across all epochs, it may indicate:
*   Not enough training time (too few epochs or batches)
*   A learning rate that is too small
(low hidden size or too few layers)
*   If your predictions overshoot wildly or fluctuate too much, it may indicate:
*   Learning rate is too high
*   Overfitting or poor generalization
*   A model that is too simple for the task

Use this plot to understand how well your model is learning. Try adjusting hyperparameters and observe how the prediction curves change over epochs

### Tuning the Model: Try It Yourself

Use the sliders below to adjust the most important hyperparameters of your LSTM model. After you set the values, the model will automatically retrain and show updated predictions and evaluation metrics.

#### What Each Slider Controls:

| **Slider**   | **Meaning** |
|--------------|-------------|
| **Hidden**   | The number of hidden units in the LSTM. Higher values allow the model to learn more complex patterns but can lead to overfitting or longer training time. Try 32 to 128. |
| **Layers**   | Number of stacked LSTM layers. More layers can capture more abstract features, but may also make training harder. Start with 1 or 2. |
| **LR (Learning Rate)** | Controls how fast the model updates its weights. Too low and training is slow; too high and the model may diverge. Typical range: `0.0005` to `0.01`. |
| **Batch**    | Number of examples used in one training update. Small batches give more noisy feedback but allow quicker updates. Common range: `16–64`. |
| **Epochs**   | Number of complete passes through the training data. More epochs allow the model to refine predictions — but watch for overfitting. |

---

#### Tips for Exploration:

- Start with the default values and observe the output R² score and predicted vs actual plot.
- Try lowering the hidden size to 32 — does performance drop?
- Increase epochs to 30 — how does prediction accuracy improve?
- Set a high learning rate (e.g., 0.01) — does the model overshoot?

> Your goal is to **learn how different hyperparameters affect the model’s ability to predict stock prices.** Don’t just aim for the highest score — aim to understand the system’s behavior.

In [ ]:
# Interactive hyperparameter tuning widget
hidden_slider = widgets.IntSlider(value=64, min=16, max=128, step=16, description='Hidden:')
layers_slider = widgets.IntSlider(value=2, min=1, max=4, description='Layers:')
lr_slider = widgets.FloatSlider(value=0.001, min=0.0001, max=0.01, step=0.0005, description='LR:')
batch_slider = widgets.IntSlider(value=32, min=8, max=128, step=8, description='Batch:')
epoch_slider = widgets.IntSlider(value=10, min=1, max=50, step=1, description='Epochs:')

def run_all(hidden, layers, LR, Batch, Epochs):
    model = train_model(hidden, layers, LR, Batch, Epochs)
    evaluate_model(model)

ui = widgets.VBox([hidden_slider, layers_slider, lr_slider, batch_slider, epoch_slider])
out = widgets.interactive_output(run_all, {
    'hidden': hidden_slider,
    'layers': layers_slider,
    'LR': lr_slider,
    'Batch': batch_slider,
    'Epochs': epoch_slider
})

display(ui, out)

---

## What This Notebook Prepares You For

This exercise helps you build intuition for:

- How RNN-based models (like LSTM) learn to make predictions over time
- How input features (Open, High, Volume) contribute to output predictions
- How to tune your model to improve accuracy
- How to measure model performance using **R² score**, just like in the Bitcoin RNN lab

---

### Ready for Unit 4 Lab RNN?

When you switch to the Bitcoin dataset:
- Use only the `High`, `Low`, and `Open` columns as input
- Use `Close` as the target
- You must achieve `r2score > 0.8` for credit
- Your final file must be named `rnn.py`
- Follow the same structure: load, scale, split, tensorize, train, predict, report `r2score`

This notebook is your blueprint — understand it fully, and you'll be ready to succeed in the graded lab.